# Phase 3: Baseline ladder (feature-only) under an honest CV harness

This notebook establishes **how hard the problem is and what a learned model must
beat**, before any typewell matching (that is notebook 4). Two design commitments
drive everything here:

1. **The eval region is a contiguous forward tail.** The competition hides the
   end of each well and asks you to predict it from the head. So every score in
   this notebook hides the last *k%* of each well and scores **only** those
   hidden rows. We sweep *k* to see sensitivity.
2. **Score under two CV schemes, side by side.** A *naive* random KFold (the
   optimistic one that ignores well/pad structure) and an *honest* pad-grouped
   scheme. The **gap between them is a diagnostic**: if naive looks great and
   honest doesn't, that gap is the likely reason earlier scores didn't hold up.

### The ladder (this notebook = rungs 0 and 1)
- **Rung 0 — dead-reckoning floor.** No learning: carry the last known TVT value
  flat across the eval tail. EDA showed the lateral flattens in TVT near the toe,
  so "hold last value" is the honest floor (slope extrapolation overshoots badly).
- **Rung 1 — feature regressors.** Predict **ΔTVT** (per the EDA target decision)
  from the cleaned features + flags, every available regressor, across the mask
  sweep and both CV schemes.
- **Rung 2 — typewell GR matching.** Deferred to notebook 4 (the headline signal).

Metric: **RMSE on TVT** (the competition metric), always in absolute TVT space
even for models that predict ΔTVT, so all rungs are directly comparable.

## Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

SRC_DIR = Path("../src")
sys.path.insert(0, str(SRC_DIR.resolve()))
from rogii_wellbore import clean  # noqa: E402

CLEAN_DIR = Path("../data/interim/clean")
INTERIM_DIR = Path("../data/interim")
CONFIG_PATH = INTERIM_DIR / "clean_config.json"
PAD_GROUPS = INTERIM_DIR / "well_pad_groups.parquet"

cfg = clean.load_config(CONFIG_PATH)
TARGET = cfg["schema"]["target"]  # "TVT"
PARAM = cfg["target"]["recommended_parameterization"]  # "delta"
print("target:", TARGET, "| parameterization:", PARAM)
print("forbidden features:", cfg["schema"]["forbidden_features"])

target: TVT | parameterization: delta
forbidden features: ['ANCC', 'ASTNL', 'ASTNU', 'BUDA', 'EGFDL', 'EGFDU', 'TVT']


## Load cleaned training data

Only the train split has truth `TVT`, so all CV scoring happens here. We load
every cleaned per-well horizontal file. (5M rows — a couple of minutes.)

In [2]:
def load_clean_train() -> pd.DataFrame:
    files = sorted((CLEAN_DIR / "train").glob("*__horizontal_well.csv"))
    parts = []
    for p in files:
        df = pd.read_csv(p)
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


train = load_clean_train()
print("rows:", f"{len(train):,}", "| wells:", train["well_id"].nunique())

# attach pad groups for the honest CV scheme
try:
    pads = pd.read_parquet(PAD_GROUPS)
except Exception:
    pads = pd.read_csv(str(PAD_GROUPS).replace(".parquet", ".csv"))
train = train.merge(pads, on="well_id", how="left")
print("pad groups attached:", train["pad_id"].nunique(), "pads")
assert train[TARGET].notna().all(), "train TVT should be fully observed"

rows: 5,092,255 | wells: 767
pad groups attached: 625 pads


## The CV harness

Three pieces, deliberately small and inspectable:

- `tail_mask(n, frac)` — boolean per-row mask, `True` = hidden eval tail (the last
  `frac` of the well by MD order).
- fold assignment — `naive` (random KFold over wells) vs `pad` (GroupKFold on
  `pad_id`). Both split at the **well level** so a well is never partly in train
  and partly in validation; the difference is only whether co-located wells are
  kept together (pad) or can be separated (naive).
- `score_fold` — train on the **known head** of training-fold wells, predict the
  **hidden tail** of validation-fold wells, RMSE on TVT over hidden rows only.

In [3]:
from sklearn.model_selection import GroupKFold, KFold


def tail_mask(n: int, frac: float) -> np.ndarray:
    k = round(n * frac)
    m = np.zeros(n, dtype=bool)
    if k > 0:
        m[n - k :] = True
    return m


def well_folds(wells, groups_map, scheme, n_splits=5, seed=0):
    # Yield (train_wells, val_wells) at the WELL level.
    # scheme='naive' -> random KFold over wells (ignores pads).
    # scheme='pad'   -> GroupKFold so co-located wells stay together.
    wells = np.array(sorted(wells))
    if scheme == "naive":
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, va in kf.split(wells):
            yield wells[tr], wells[va]
    elif scheme == "pad":
        groups = np.array([groups_map[w] for w in wells])
        gkf = GroupKFold(n_splits=n_splits)
        for tr, va in gkf.split(wells, groups=groups):
            yield wells[tr], wells[va]
    else:
        raise ValueError(scheme)


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

## Target construction: ΔTVT with reconstruction

Models predict per-row ΔTVT (change from the previous row). To score in absolute
TVT we **reconstruct**: walk the eval tail forward from the last known TVT,
adding predicted deltas. This is what makes a delta model comparable to the
flat-hold floor on the competition's TVT-RMSE metric.

In [4]:
FEATURES = ["GR_z", "gr_missing", "gr_flatline", "traj_teleport", "MD", "X", "Y", "Z"]
# leakage guard: none of these may be a train-only column
clean.assert_no_leakage(FEATURES, cfg)
print("features:", FEATURES)


def make_delta(df_well: pd.DataFrame) -> np.ndarray:
    return df_well[TARGET].diff().fillna(0.0).values


def reconstruct_tvt(df_well, mask, delta_pred):
    # Given predicted per-row deltas on eval rows, rebuild absolute TVT by
    # integrating forward from the last known TVT value.
    tvt = df_well[TARGET].values.astype(float)
    out = tvt.copy()
    known_idx = np.where(~mask)[0]
    last = known_idx[-1]
    running = tvt[last]
    for i in np.where(mask)[0]:
        running = running + delta_pred[i]
        out[i] = running
    return out

features: ['GR_z', 'gr_missing', 'gr_flatline', 'traj_teleport', 'MD', 'X', 'Y', 'Z']


## Rung 0 — dead-reckoning floor (no learning)

Carry the last known TVT flat across the eval tail. EDA established this beats
slope extrapolation (the lateral flattens near the toe). Every learned model must
beat this to justify itself.

In [5]:
def floor_hold_last(df_well, mask):
    tvt = df_well[TARGET].values.astype(float)
    last = np.where(~mask)[0][-1]
    pred = tvt.copy()
    pred[mask] = tvt[last]
    return pred


def eval_floor(train, mask_frac):
    errs = []
    for _wid, g in train.groupby("well_id", sort=False):
        g = g.sort_values("MD")
        m = tail_mask(len(g), mask_frac)
        pred = floor_hold_last(g, m)
        errs.append(rmse(pred[m], g[TARGET].values[m]))
    return float(np.mean(errs))


MASK_FRACS = [0.10, 0.25, 0.40]
floor_rows = []
for f in MASK_FRACS:
    r = eval_floor(train, f)
    floor_rows.append({"model": "floor_hold_last", "cv": "per_well", "mask_frac": f, "rmse_tvt": r})
    print(f"floor RMSE @ mask {f:.0%}: {r:.3f} ft")
floor_df = pd.DataFrame(floor_rows)

floor RMSE @ mask 10%: 5.297 ft
floor RMSE @ mask 25%: 8.745 ft
floor RMSE @ mask 40%: 10.508 ft


## Rung 1 — feature regressors (ΔTVT), full grid

Every available regressor × mask fraction × CV scheme. Gradient-boosting libs are
optional — the registry detects what is installed and skips the rest, so this
runs anywhere.

**Why this is the meaningful grid, not hyperparameter permutations:** the goal is
to rank *approaches* and expose the naive-vs-honest CV gap, not to tune one model.
Each regressor uses a sensible fixed config.

In [6]:
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge


def build_registry():
    reg = {}
    reg["ridge"] = lambda: Ridge(alpha=1.0)
    reg["random_forest"] = lambda: RandomForestRegressor(
        n_estimators=200, max_depth=None, n_jobs=-1, random_state=0
    )
    reg["extra_trees"] = lambda: ExtraTreesRegressor(n_estimators=200, n_jobs=-1, random_state=0)
    reg["hist_gbm"] = lambda: HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, random_state=0
    )
    reg["sk_gbm"] = lambda: GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, random_state=0
    )
    try:
        import lightgbm as lgb

        reg["lightgbm"] = lambda: lgb.LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=63,
            n_jobs=-1,
            random_state=0,
            verbose=-1,
        )
    except ImportError:
        print("lightgbm not installed - skipping")
    try:
        import xgboost as xgb

        reg["xgboost"] = lambda: xgb.XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            n_jobs=-1,
            random_state=0,
            verbosity=0,
        )
    except ImportError:
        print("xgboost not installed - skipping")
    return reg


REGISTRY = build_registry()

# Runtime control. The full grid is (n_models x 3 masks x 2 schemes x 5 folds)
# fits over 773 wells; random_forest and sk_gbm are the slow ones (200 trees each).
# Set FAST=True for a quick pass with the fast learners only, then FAST=False for
# the complete comparison once you are happy the harness behaves.
FAST = True
FAST_MODELS = ["ridge", "hist_gbm", "lightgbm", "xgboost"]  # drop slow RF / sk_gbm
if FAST:
    REGISTRY = {k: v for k, v in REGISTRY.items() if k in FAST_MODELS}
print("regressors:", list(REGISTRY.keys()), "| FAST =", FAST)

regressors: ['ridge', 'hist_gbm', 'lightgbm', 'xgboost'] | FAST = True


In [7]:
def build_xy(train, wells, mask_frac, fit_region):
    # Assemble (X, y) rows. fit_region='known' uses only un-masked head rows
    # (what we train on); 'eval' returns the masked tail rows (what we predict).
    # y is ΔTVT.
    Xs, ys, _idx = [], [], []
    sub = train[train["well_id"].isin(set(wells))]
    for _wid, g in sub.groupby("well_id", sort=False):
        g = g.sort_values("MD")
        m = tail_mask(len(g), mask_frac)
        delta = make_delta(g)
        take = ~m if fit_region == "known" else m
        Xs.append(g[FEATURES].values[take])
        ys.append(delta[take])
    if not Xs:
        return np.empty((0, len(FEATURES))), np.empty((0,))
    return np.vstack(Xs), np.concatenate(ys)


def predict_eval_tvt(model, train, wells, mask_frac):
    # Predict ΔTVT on eval tails, reconstruct TVT, return (pred, true) over
    # eval rows pooled across the given wells.
    preds, trues = [], []
    sub = train[train["well_id"].isin(set(wells))]
    for _wid, g in sub.groupby("well_id", sort=False):
        g = g.sort_values("MD").reset_index(drop=True)
        m = tail_mask(len(g), mask_frac)
        dpred = np.zeros(len(g))
        dpred[m] = model.predict(g[FEATURES].values[m])
        tvt_pred = reconstruct_tvt(g, m, dpred)
        preds.append(tvt_pred[m])
        trues.append(g[TARGET].values[m])
    return np.concatenate(preds), np.concatenate(trues)

In [8]:
def run_grid(train, mask_fracs, schemes, n_splits=5, seed=0):
    train = train.copy()
    train["well_id"] = train["well_id"].astype(
        str
    )  # all-digit ids parse as int and collide with hex ids on sort
    wells = sorted(train["well_id"].unique())
    groups_map = train.drop_duplicates("well_id").set_index("well_id")["pad_id"].to_dict()
    rows = []
    for scheme in schemes:
        for frac in mask_fracs:
            for name, make in REGISTRY.items():
                fold_rmses = []
                for tr_w, va_w in well_folds(wells, groups_map, scheme, n_splits, seed):
                    Xtr, ytr = build_xy(train, tr_w, frac, "known")
                    model = make()
                    model.fit(Xtr, ytr)
                    pred, true = predict_eval_tvt(model, train, va_w, frac)
                    fold_rmses.append(rmse(pred, true))
                rows.append(
                    {
                        "model": name,
                        "cv": scheme,
                        "mask_frac": frac,
                        "rmse_tvt": float(np.mean(fold_rmses)),
                        "rmse_std": float(np.std(fold_rmses)),
                    }
                )
                print(
                    f"[{scheme:5s}] {name:14s} mask {frac:.0%}: "
                    f"RMSE {np.mean(fold_rmses):.3f} +/- {np.std(fold_rmses):.3f}"
                )
    return pd.DataFrame(rows)


# WARNING: full grid over 773 wells x all regressors x 3 masks x 2 schemes x 5
# folds is the expensive cell. RandomForest/sk_gbm are the slow ones.
grid = run_grid(train, MASK_FRACS, schemes=["naive", "pad"], n_splits=5)

[naive] ridge          mask 10%: RMSE 1139.176 +/- 504.937
[naive] hist_gbm       mask 10%: RMSE 2379.602 +/- 2441.011
[naive] lightgbm       mask 10%: RMSE 4547.428 +/- 3550.395
[naive] xgboost        mask 10%: RMSE 6951.120 +/- 5810.686
[naive] ridge          mask 25%: RMSE 2796.238 +/- 1224.944
[naive] hist_gbm       mask 25%: RMSE 4845.920 +/- 5529.887
[naive] lightgbm       mask 25%: RMSE 3598.009 +/- 2980.153
[naive] xgboost        mask 25%: RMSE 5862.275 +/- 4249.884
[naive] ridge          mask 40%: RMSE 5404.505 +/- 2383.911
[naive] hist_gbm       mask 40%: RMSE 2690.473 +/- 2095.369
[naive] lightgbm       mask 40%: RMSE 3810.637 +/- 2994.217
[naive] xgboost        mask 40%: RMSE 11176.968 +/- 12847.791
[pad  ] ridge          mask 10%: RMSE 1118.179 +/- 483.687
[pad  ] hist_gbm       mask 10%: RMSE 7955.043 +/- 5529.204
[pad  ] lightgbm       mask 10%: RMSE 5481.032 +/- 4552.508
[pad  ] xgboost        mask 10%: RMSE 6262.885 +/- 4446.746
[pad  ] ridge          mask 25%: RMSE 27

## Results — comparison tables

In [9]:
results = pd.concat([floor_df.assign(rmse_std=0.0), grid], ignore_index=True)
results = results.sort_values(["mask_frac", "cv", "rmse_tvt"]).reset_index(drop=True)

# Pivot: model x (cv, mask_frac) RMSE
pivot = results.pivot_table(index="model", columns=["cv", "mask_frac"], values="rmse_tvt")
print("RMSE on TVT (ft) — lower is better:")
pivot.round(3)

RMSE on TVT (ft) — lower is better:


cv                  naive                            pad                      per_well               
mask_frac            0.10      0.25       0.40      0.10      0.25       0.40     0.10   0.25    0.40
model                                                                                                
floor_hold_last       NaN       NaN        NaN       NaN       NaN        NaN    5.297  8.745  10.508
hist_gbm         2379.602  4845.920   2690.473  7955.043  5316.829   4865.368      NaN    NaN     NaN
lightgbm         4547.428  3598.009   3810.637  5481.032  9281.734  16452.296      NaN    NaN     NaN
ridge            1139.176  2796.238   5404.505  1118.179  2742.617   5320.564      NaN    NaN     NaN
xgboost          6951.120  5862.275  11176.968  6262.885  6723.650  11791.666      NaN    NaN     NaN

In [10]:
# The headline diagnostic: naive vs honest gap per model (at the middle mask).
mid = 0.25
piv2 = results[results["mask_frac"] == mid].pivot_table(
    index="model", columns="cv", values="rmse_tvt"
)
if {"naive", "pad"}.issubset(piv2.columns):
    piv2["honest_minus_naive"] = piv2["pad"] - piv2["naive"]
    piv2["optimism_pct"] = piv2["honest_minus_naive"] / piv2["naive"] * 100
    print(f"Naive vs honest (pad) CV @ mask {mid:.0%}:")
    print(piv2.sort_values("pad").round(3))
    print("\nA large positive 'honest_minus_naive' = naive CV was flattering the model.")

Naive vs honest (pad) CV @ mask 25%:
cv                  naive       pad  per_well  honest_minus_naive  optimism_pct
model                                                                          
ridge            2796.238  2742.617       NaN             -53.621        -1.918
hist_gbm         4845.920  5316.829       NaN             470.910         9.718
xgboost          5862.275  6723.650       NaN             861.375        14.694
lightgbm         3598.009  9281.734       NaN            5683.725       157.969
floor_hold_last       NaN       NaN     8.745                 NaN           NaN

A large positive 'honest_minus_naive' = naive CV was flattering the model.


In [11]:
# Save results for notebook 4 to build on.
out = INTERIM_DIR / "baseline_results.csv"
results.to_csv(out, index=False)
print("saved:", out)

# Best honest model vs floor at each mask.
print("\nBest honest (pad-CV) model vs floor:")
for f in MASK_FRACS:
    fl = results[(results.model == "floor_hold_last") & (results.mask_frac == f)]["rmse_tvt"].iloc[
        0
    ]
    sub = results[
        (results.cv == "pad") & (results.mask_frac == f) & (results.model != "floor_hold_last")
    ]
    if len(sub):
        best = sub.sort_values("rmse_tvt").iloc[0]
        verdict = "beats floor" if best.rmse_tvt < fl else "WORSE than floor"
        print(
            f"  mask {f:.0%}: floor {fl:.3f} | best {best.model} {best.rmse_tvt:.3f} -> {verdict}"
        )

saved: ..\data\interim\baseline_results.csv

Best honest (pad-CV) model vs floor:
  mask 10%: floor 5.297 | best ridge 1118.179 -> WORSE than floor
  mask 25%: floor 8.745 | best ridge 2742.617 -> WORSE than floor
  mask 40%: floor 10.508 | best hist_gbm 4865.368 -> WORSE than floor


## Reading these results

> **Expect the flat floor to be hard to beat.** Because the learned rung predicts
> per-row ΔTVT and integrates it across the eval tail, prediction error
> *accumulates* with distance from the last known row — while `floor_hold_last`
> never drifts. If the regressors lose to the floor (especially at larger mask
> fractions), that is not a harness bug: it is direct evidence that **naive
> per-row delta regression is the wrong framing**, and that the typewell match
> (notebook 4), which re-anchors against an absolute reference, is where real
> gains live. This is a likely explanation for earlier underperformance.

**The two questions this notebook answers:**

1. *Do the feature regressors beat the no-learning floor?* If the best pad-CV
   model barely beats `floor_hold_last`, that is the real message: features alone
   (trajectory + GR statistics, no typewell) carry little signal for the forward
   tail, and the **typewell matching in notebook 4 is where the gains must come
   from**. That would be consistent with the EDA, where the headline signal was
   GR-vs-typewell alignment, not raw features.

2. *Was earlier optimism a CV artifact?* The `honest_minus_naive` column. A large
   gap means random CV let co-located/structurally-similar wells leak across
   folds, inflating scores — the likely culprit behind earlier results that did
   not hold up.

**For notebook 4:** reuse this harness verbatim (`tail_mask`, `well_folds`,
`reconstruct_tvt`, `rmse`). Add the typewell-matching predictor as a new rung and
score it in the same grid so it is directly comparable to these baselines. The
bar to beat is the **best pad-CV RMSE** here, and the floor is `floor_hold_last`.